# Generate verdicts and build report.json

The **G** in RAG. For each SRS requirement, an LLM (Gemini) reads the requirement, the NASA clauses
retrieved in `07_retrieve_relevant_clauses.ipynb`, and a few rules that always apply, then writes a
verdict. Lauren's models in `report_schema.py` check every verdict, and the verdicts are assembled
into `data/report.json` and checked with Mark's `validate_report.py`.

**Who decides what.** The LLM decides the status, the issue codes, the quotes, and which clauses
support each issue, citing clauses by ID. The code fills in the exact NASA section and URL from
`addressable_clauses.csv`, so a citation can't be made up.

**Setup.** Put `GOOGLE_API_KEY=...` in a `.env` file at the repo root (git ignores it). Answers are
cached in `data/llm_cache/`, so re-running only calls the API for requirements that haven't been
reviewed yet; if a run stops (e.g. Gemini is busy), just run it again. Delete a file there to review
that requirement again.

In [1]:
import json, sys, time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "rag_retrievals.csv").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "contract"))
from review_tools import (ALWAYS_INCLUDE, SYSTEM, DraftReview, build_report, clauses, clauses_for, fallback,
                          prompt_for, retrievals, srs, to_review, with_api_retries)
from validate_report import validate

load_dotenv(ROOT / ".env")

MODEL = "gemini-3.5-flash"  # the API serves this name as gemini-3.6-flash; kept as is so the answer cache stays valid
PROMPT_VERSION = 2  # bump when the prompt changes, so old and new answers are never mixed
CACHE_DIR = ROOT / "data" / "llm_cache" / f"{MODEL}_prompt{PROMPT_VERSION}"
REPORT_PATH = ROOT / "data" / "report.json"
MAX_ATTEMPTS = 3    # tries per requirement before falling back to UNDETERMINED
PAUSE_SECONDS = 4   # between API calls, to stay under the free tier's rate limit

In [2]:
benchmark = pd.read_csv(ROOT / "data" / "benchmark" / "benchmark_inputs_for_teams_v1.csv")
calibration = pd.read_csv(ROOT / "data" / "benchmark" / "calibration_examples_for_teams_v1.csv")

print(len(srs), "requirements |", len(clauses), "clauses |", retrievals.req_id.nunique(), "requirements with retrievals")

54 requirements | 62 clauses | 54 requirements with retrievals


## Clauses shown to the LLM

Each requirement gets its top 5 retrieved clauses plus three that always apply. Retrieval finds
clauses on the same *topic* as the requirement, so it never surfaces the general writing rules:
in 07, `swehb-1.b` ("specific, testable, and traceable") was in nobody's top 5.

| Always included | Why |
|---|---|
| `swehb-1.b-guidance` | Each requirement should be specific, testable, and traceable |
| `swehb-3.2.1-guidance` | Characteristics of functional requirements (clear, verifiable, behavioral, ...) |
| `npr-4.1.4-requirement` | SWE-184: safety constraints and controls belong in the requirements |

The whole SRS is included too (about 12 KB), so the LLM can use the open items, mode definitions,
and interfaces when judging one requirement.

In [3]:
print(clauses_for("SFMC-REQ-015"))

['npr-3.7.3-requirement', 'swehb-3.2.1.4-guidance', 'swehb-3.2.2.3-guidance', 'swehb-6.1-guidance', 'swehb-3.2.1.7-guidance', 'swehb-1.b-guidance', 'swehb-3.2.1-guidance', 'npr-4.1.4-requirement']


## What the LLM fills in, the checks, and the prompt

These live in `review_tools.py`, shared with the agent in `11_tool_calling_agent.ipynb`:

- **`DraftReview`**: a smaller version of Lauren's `RequirementReview`. The LLM doesn't type requirement IDs,
  raw text, NASA sections, or URLs; the code adds those. It writes a short `analysis` first, so it reasons
  before it decides. SWE links are limited to **SWE-184**, the only §4.1 provision whose content (safety
  constraints and controls) can appear in a requirement's text; the others are process rules an SRS can't show.
- **`to_review`**: turns a draft into a `RequirementReview`, which runs all of Lauren's checks (IDs, quotes
  found in the SRS, citations among the 62 clauses, ACCEPTABLE means no issues). Any failure is sent back to
  the LLM to fix.
- **`SYSTEM`**: the rules, from Mark's output contract and the team's quality taxonomy. They describe each
  issue type in general terms only: no benchmark requirement or answer appears in the prompt, so the
  calibration comparison at the end stays a fair check.

## Calling the LLM

`init_chat_model` keeps the provider in one string, so switching to another LLM later means changing
`MODEL` above and this line, then installing that provider's package and key.

In [4]:
def make_reviewer():
    from langchain.chat_models import init_chat_model
    llm = init_chat_model(f"google_genai:{MODEL}", max_retries=1)  # every retry counts against the free daily limit
    return llm.with_structured_output(DraftReview, method="json_schema")

reviewer = make_reviewer()

def review_requirement(row):
    """Returns (review, attempts). attempts is 0 for a cached answer and None for a fallback."""
    cache = CACHE_DIR / f"{row.req_id}.json"
    allowed = clauses_for(row.req_id)
    if cache.exists():
        return to_review(row, DraftReview.model_validate_json(cache.read_text(encoding="utf-8")), allowed), 0

    messages = [("system", SYSTEM), ("human", prompt_for(row, allowed))]
    error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        draft = None
        try:
            draft = with_api_retries(lambda: reviewer.invoke(messages))
            time.sleep(PAUSE_SECONDS)
            if draft is None:
                raise ValueError("no structured answer was returned")
            review = to_review(row, draft, allowed)
        except ValueError as e:  # includes Pydantic's ValidationError and LangChain's parsing errors
            error = e
            answer = draft.model_dump_json() if isinstance(draft, DraftReview) else "(unreadable answer)"
            messages += [("ai", answer), ("human", f"That answer failed these checks:\n{e}\n\nFix them and answer again.")]
            continue
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(draft.model_dump_json(indent=2), encoding="utf-8")
        return review, attempt
    print(f"{row.req_id}: falling back to UNDETERMINED: {error}")
    return fallback(row, f"validation failed after {MAX_ATTEMPTS} attempts"), None

## Try a few first

One good requirement, one with a missing response, and two safety cases. Check these before running
all 54.

In [5]:
TRY_FIRST = ["SFMC-REQ-002", "SFMC-REQ-010", "SFMC-REQ-015", "SFMC-REQ-035"]
attempts = {}

for _, row in srs[srs.req_id.isin(TRY_FIRST)].iterrows():
    review, attempts[row.req_id] = review_requirement(row)
    codes = ", ".join(i.issue_code for i in review.quality_issues) or "-"
    print(f"{review.requirement_id}  {review.quality_status:13} {codes:40} SWE: {review.primary_swe_id or '-'}  (attempts: {attempts[row.req_id]})")
    print(f"    {review.summary}\n")

SFMC-REQ-002  ACCEPTABLE    -                                        SWE: -  (attempts: 0)
    SFMC-REQ-002 is an acceptable requirement with a measurable timing criterion (500 ms) and clearly defined operational modes.

SFMC-REQ-010  NEEDS_REVIEW  UNSPECIFIED_RESPONSE                     SWE: SWE-184  (attempts: 0)
    SFMC-REQ-010 is a valid software requirement, but it is incomplete and unverifiable because it relies on the placeholder phrase 'take appropriate corrective action' to describe fault recovery logic.

SFMC-REQ-015  NEEDS_REVIEW  UNDEFINED_TERM                           SWE: SWE-184  (attempts: 0)
    SFMC-REQ-015 specifies a safety interlock to prevent firing at friendly spacecraft, but it relies on an undefined classification category that is marked as an open item in §11.

SFMC-REQ-035  NEEDS_REVIEW  SCOPE_AMBIGUITY, SAFETY_INTERACTION      SWE: SWE-184  (attempts: 0)
    SFMC-REQ-035 defines a pilot safety override function, but is incomplete. It suffers from scope am

## Review all 54

In [6]:
reviews = []
for _, row in srs.iterrows():
    review, a = review_requirement(row)
    attempts.setdefault(row.req_id, a)  # keep the count from the try-first cell
    reviews.append(review)

fallbacks = [rid for rid, a in attempts.items() if a is None]
print(len(reviews), "reviewed |", sum(a == 0 for a in attempts.values()), "from cache |",
      sum(bool(a) and a > 1 for a in attempts.values()), "needed a retry |", len(fallbacks), "fell back")
pd.Series([r.quality_status for r in reviews]).value_counts()

54 reviewed | 54 from cache | 0 needed a retry | 0 fell back


NEEDS_REVIEW    30
ACCEPTABLE      24
Name: count, dtype: int64

## Assemble and validate report.json

Only `requirement_reviews` is generated so far, so the run is `DEMO` scope. A `FULL_SRS` run also
needs the six SWE summaries and the document findings.

In [7]:
limitations = [
    "Requirement reviews only; relationships, document findings, and SWE coverage are not generated yet.",
    "No NASA project compliance is determined from the SRS alone.",
    "Verdicts are LLM-generated and have not been checked by a human reviewer.",
]
if fallbacks:
    limitations.append(f"{len(fallbacks)} requirement(s) failed automated review and are marked UNDETERMINED: {', '.join(fallbacks)}.")

method = (f"RAG: top-5 clauses retrieved from Chroma (all-MiniLM-L6-v2) plus {len(ALWAYS_INCLUDE)} always-included "
          f"clauses; verdicts by {MODEL} (served by the API as gemini-3.6-flash) with structured output; "
          "citations filled from addressable_clauses.csv; validated with report_schema.py")
build_report(reviews, method, limitations, REPORT_PATH)
print("validate_report.py counts (reviews, relationships, document findings, SWEs):", validate(REPORT_PATH))

validate_report.py counts (reviews, relationships, document findings, SWEs): (54, 0, 0, 0)


## Compare with the calibration examples

Mark's 7 requirement-level calibration answers. A match on status and on the expected issue code is
the target; differences are worth reading, since some calls are judgment calls.

In [8]:
by_id = {r.requirement_id: r for r in reviews}
cal = calibration[calibration.scope == "REQUIREMENT"][["target_ids", "expected_outcome", "issue_code"]].copy()
cal["predicted"] = cal.target_ids.map(lambda i: by_id[i].quality_status)
cal["predicted_codes"] = cal.target_ids.map(lambda i: ", ".join(q.issue_code for q in by_id[i].quality_issues) or "NONE")
cal["status_ok"] = cal.predicted == cal.expected_outcome
cal["code_ok"] = [code in codes.split(", ") for code, codes in zip(cal.issue_code, cal.predicted_codes)]
print(f"status {cal.status_ok.sum()}/{len(cal)} | issue code {cal.code_ok.sum()}/{len(cal)}")
cal

status 7/7 | issue code 7/7


,target_ids,expected_outcome,issue_code,predicted,predicted_codes,status_ok,code_ok
0,SFMC-REQ-002,ACCEPTABLE,NONE,ACCEPTABLE,NONE,True,True
1,SFMC-REQ-006,NEEDS_REVIEW,VAGUE_CRITERION,NEEDS_REVIEW,VAGUE_CRITERION,True,True
2,SFMC-REQ-010,NEEDS_REVIEW,UNSPECIFIED_RESPONSE,NEEDS_REVIEW,UNSPECIFIED_RESPONSE,True,True
3,SFMC-REQ-021,NEEDS_REVIEW,VAGUE_CRITERION,NEEDS_REVIEW,"VAGUE_CRITERION, UNSPECIFIED_RESPONSE, UNDEFIN...",True,True
4,SFMC-REQ-035,NEEDS_REVIEW,SAFETY_INTERACTION,NEEDS_REVIEW,"SCOPE_AMBIGUITY, SAFETY_INTERACTION",True,True
5,SFMC-REQ-037,NEEDS_REVIEW,VAGUE_CRITERION,NEEDS_REVIEW,VAGUE_CRITERION,True,True
6,SFMC-REQ-043,NEEDS_REVIEW,ESCAPE_CLAUSE,NEEDS_REVIEW,ESCAPE_CLAUSE,True,True


## Checks

In [9]:
bench_reqs = set(benchmark.loc[benchmark.scope == "REQUIREMENT", "target_ids"])
assert len(reviews) == len(srs) == 54, "every requirement gets exactly one review"
assert bench_reqs <= set(by_id), f"benchmark requirements missing: {sorted(bench_reqs - set(by_id))}"
assert json.loads(REPORT_PATH.read_text(encoding="utf-8"))["requirement_reviews"], "report.json was written"
print(f"all {len(bench_reqs)} benchmark requirements reviewed; report.json passes validate_report.py")

all 23 benchmark requirements reviewed; report.json passes validate_report.py
